# Fracture Classification with Datamint and FracAtlas Dataset

This notebook demonstrates how to build an end-to-end binary classification pipeline using **Datamint** and the **FracAtlas** dataset.

## Overview

You will learn how to:
- **Set up a Datamint project** for managing medical imaging data
- **Upload the FracAtlas dataset** to Datamint
- **Train a model** using the Datamint Trainer API — one line of code handles dataset loading, model creation, training, and experiment tracking
- **Deploy the model** for inference using Datamint's model serving

## Comparison with the Manual Workflow

The conventional way requires you to:
1. Define transforms, a Dataset subclass, and DataLoaders (~80 lines)
2. Define a full LightningModule with loss, metrics, and optimizer (~80 lines)
3. Configure MLflow logger, callbacks, and Trainer (~30 lines)

With the Trainer API, **all of that is replaced by ~3 lines**:

```python
from datamint.lightning import EfficientNetV2Trainer

trainer = EfficientNetV2Trainer(project='FracAtlas')
results = trainer.fit()
```

## Table of Contents

1. [Setup: Create Project and Upload Dataset](#1-setup-create-project-and-upload-dataset)
2. [Training with EfficientNetV2Trainer](#2-training-with-efficientnetv2trainer)
3. [Local Inference](#3-local-inference)
4. [Custom Model Architecture](#4-custom-model-architecture)
5. [Deployment](#5-deployment)

## Required Dependencies

```bash
pip install datamint
```

## Dataset Overview

The **FracAtlas** dataset contains 4,083 musculoskeletal radiographs annotated for fracture detection, localization, and segmentation. In this notebook, we focus on **binary classification**: determining whether an X-ray image shows a fracture or not.

In [1]:
from datamint import Api

PROJECT_NAME = "FracAtlas"
api = Api()

## 1. Setup: Create Project and Upload Dataset

In this section, we will:
- Create a new Datamint project (or retrieve an existing one)
- Download the FracAtlas dataset from Figshare
- Upload images to Datamint with appropriate tags
- Create classification annotations for each image

In [2]:
proj = api.projects.create(
    name=PROJECT_NAME,
    description="Project to train a binary classification model on FracAtlas dataset",
    exists_ok=True # Just return the existing project if it already exists
)
proj # Display project information

FracAtlas
2025-12-03T02:09:00.363Z
datamint-dev@mail.com
False
4083
0
Project to train a binary classification model on FracAtlas dataset


### 1.1 Download FracAtlas Dataset

The FracAtlas dataset is publicly available on Figshare. We'll download and extract it programmatically.

**Dataset Source:** [Figshare Repository](https://doi.org/10.6084/m9.figshare.22363012)

**Citation:** 
> Abedeen, I., et al. (2023). FracAtlas: A Dataset for Fracture Classification, Localization and Segmentation of Musculoskeletal Radiographs. Scientific Data, 10(1). doi:10.1038/s41597-023-02432-4

> [!Note]
> The download may take a few minutes depending on your internet connection (~1.2 GB compressed).


In [ ]:
import requests
import zipfile
import os

# Retrieve and download FracAtlas dataset from Figshare
# It might take a while depending on your internet connection. ~50 seconds on a 100Mbps connection
r = requests.get('https://api.figshare.com/v2/articles/22363012')
if r.status_code == 200:
    file_metadata = r.json()['files'][0]
    file_name = file_metadata['name']
    print(f'Downloading {file_name}...')
    
    # Download and extract
    r = requests.get(file_metadata['download_url'], allow_redirects=True)
    with open(file_name, 'wb') as f:
        f.write(r.content)
    
    print(f'Unzipping {file_name}...')
    with zipfile.ZipFile(file_name, 'r') as zip_ref:
        zip_ref.extractall(os.path.splitext(file_name)[0])
else:
    print('Error:', r.text)

### 1.2 Dataset Structure

The extracted dataset has the following structure:

```
FracAtlas/
├── images/
│   ├── Fractured/          # 717 images with visible fractures
│   │   ├── IMG0000110.jpg
│   │   └── ...
│   └── Non_fractured/      # 3,366 images without fractures
│       ├── IMG0002341.jpg
│       └── ...
└── ...
```

For this binary classification task, we'll use only the `images` folder, treating:
- `Fractured/` → Positive class (label: `has_fracture: yes`)
- `Non_fractured/` → Negative class (label: `has_fracture: no`)

In [ ]:
from pathlib import Path
import os

# get all non-fractured images
non_fractured_root_path = Path('FracAtlas/FracAtlas/images/Non_fractured/')
fractured_root_path = Path('FracAtlas/FracAtlas/images/Fractured/')
non_fractured_images_paths = [str(non_fractured_root_path / img) for img in os.listdir(non_fractured_root_path)]
fractured_images_paths = [str(fractured_root_path / img) for img in os.listdir(fractured_root_path)]

print(f'Found {len(non_fractured_images_paths)} non-fractured images')
print(f'Found {len(fractured_images_paths)} fractured images')

In [ ]:
# Upload non-fractured images to Datamint
# Tags are optional and arbitrary, but help organize and filter resources later:
# 'fracatlas' - identifies dataset source
# 'non-fractured' - identifies class for annotation creation
new_resources_list = api.resources.upload_resources(non_fractured_images_paths,
                                                    tags=['fracatlas', 'non-fractured'],
                                                    publish_to=proj,  # associate the resources to the project
                                                    progress_bar=True)

In [ ]:
# Upload fractured images to Datamint
new_resources_list = api.resources.upload_resources(fractured_images_paths,
                                                    tags=['fracatlas', 'fractured'],
                                                    publish_to=proj,  # associate the resources to the project
                                                    progress_bar=True)

### 1.3 Create Classification Annotations

Now we'll create structured annotations for each image.

In [ ]:
from tqdm.auto import tqdm

# Annotate non-fractured images with 'has_fracture: no'
nonfrac_resources_list = api.resources.get_list(project_name=PROJECT_NAME,
                                                tags=['non-fractured'])
for res in tqdm(nonfrac_resources_list):
    api.annotations.create_image_classification(resource=res,
                                                identifier='has_fracture',
                                                value='no')
# Annotate fractured images with 'has_fracture: yes'
frac_resources_list = api.resources.get_list(project_name=PROJECT_NAME,
                                             tags=['fractured'])
for res in tqdm(frac_resources_list):
    api.annotations.create_image_classification(resource=res,
                                                identifier='has_fracture',
                                                value='yes')

In [ ]:
# Inspect a sample resource to verify the upload
frac_resources_list[0]

In [ ]:
# Verify that annotations were created correctly
# The annotation should show identifier='has_fracture' and value='yes' or 'no'
api.annotations.get_list(resource=frac_resources_list[0])[0].asdict()

### 1.4 Create Train/Validation/Test Splits

We use `build_dataset` to load the project and `.split()` to create reproducible splits. Calling `.save()` persists the assignments to the server — any later session can reload the same split with `dataset.split()` (no arguments).

| Split | Percentage | Purpose |
|-------|------------|---------|
| Train | 80% | Model training |
| Validation | 10% | Hyperparameter tuning, early stopping |
| Test | 10% | Final model evaluation |

In [ ]:
from datamint.dataset import build_dataset

dataset = build_dataset(
    PROJECT_NAME,
    include_unannotated=False,
    image_categories_merge_strategy='mode',
    allow_external_annotations=True,
)

parts = dataset.split(train=0.8, val=0.1, test=0.1, seed=123)
parts.save()  

for name, ds in parts.items():
    print(f'{name:6s}: {len(ds):4d} resources')

## 2. Training with EfficientNetV2Trainer

This is where the **Trainer API** shines. Instead of manually defining:
- A `Dataset` subclass and DataLoaders
- A `LightningModule` with loss, metrics, and optimizer
- Callbacks, loggers, and a Lightning `Trainer`
- A deployment adapter

...you simply create an `EfficientNetV2Trainer` and call `fit()`.

The trainer automatically:
1. Builds an `ImageDataset` and reads the saved project-scoped splits
2. Creates image augmentation pipelines (resize, flip, color jitter, normalize)
3. Instantiates **EfficientNetV2-S** pretrained on ImageNet at 384×384
4. Sets up **CrossEntropy loss**, **Accuracy** and **F1** metrics
5. Configures MLflow logging, checkpointing, and early stopping
6. Trains, evaluates on the test split, and **registers the best checkpoint** in the MLflow Model Registry

In [ ]:
from datamint.lightning import EfficientNetV2Trainer
import os
os.environ['MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR'] = 'false'

trainer = EfficientNetV2Trainer(
    project=PROJECT_NAME,
    image_size=384,
    batch_size=8,
    max_epochs=10,
    accelerator='auto',
    # register_model_name='MyModelName'  # defaults to project name
)

results = trainer.fit()

In [ ]:
print("Test results:")
for metric_dict in results['test_results']:
    for k, v in metric_dict.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
# Open the Datamint project dashboard to view experiments and metrics
proj.show()

## 3. Local Inference

The trainer automatically registers a deployment-ready model in MLflow after training.
Load it for local prediction:

In [ ]:
from datamint.mlflow import flavors as datamint_flavor

# Load the registered model
model_loaded = datamint_flavor.load_model(f'models:/{PROJECT_NAME}/latest')

# Predict on a resource from the dataset
r = trainer.dataset[0]['resource']
prediction = model_loaded.predict([r])
print(prediction)

> [!TIP]
> You can also serve the model locally via REST API:
> ```bash
> mlflow models serve -m "models:/{PROJECT_NAME}/latest" -p 5111 --env-manager virtualenv
> ```

## 4. Custom Model Architecture

The Trainer API lets you swap in a custom backbone while keeping everything else automatic.

There are two supported patterns:

1. **(Recommended)** Pass a `ClassificationModule` subclass **class** to `model=`. Datamint manages loss, metrics, MLflow logging, and deployment compatibility. Pass the class, not an instance.
2. Pass a fully constructed `lightning.LightningModule` **instance** to `model=` when you want complete control over `training_step`, `validation_step`, `test_step`, and `configure_optimizers`.

The example below uses **ResNet-18** from torchvision as a custom backbone while keeping the full Datamint trainer workflow.

In [ ]:
from datamint.lightning import ImageClassificationTrainer
from datamint.lightning.trainers.lightning_modules import ClassificationModule
from torchvision.models import resnet18
import torch


class FracAtlasClassifier(ClassificationModule):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Replace the default timm model with a custom backbone
        self.model = resnet18(weights='DEFAULT')
        self.model.fc = torch.nn.Linear(
            self.model.fc.in_features, self.hparams['num_classes']
        )

    def forward(self, x):
        return self.model(x)

    # OPTIONAL: override test_step for custom behaviour
    # def test_step(self, batch, batch_idx):
    #     images = batch['image']              # (B, C, H, W)
    #     labels = batch['image_categories']   # (B,)
    #     # ... custom logic ...
    #     return super().test_step(batch, batch_idx)


trainer_custom = ImageClassificationTrainer(
    project=PROJECT_NAME,
    model=FracAtlasClassifier,   # pass the class, not an instance
    image_size=480,
    batch_size=8,
    max_epochs=10,
    accelerator='auto',
)

results_custom = trainer_custom.fit()

In [ ]:
print("Test results (custom model):")
for metric_dict in results_custom['test_results']:
    for k, v in metric_dict.items():
        print(f"  {k}: {v:.4f}")

### 4.1 Using a Fully Custom LightningModule

If you already have your own Lightning module, pass an instance. In this mode you own the training and evaluation logic:

```python
import lightning as L
import torch
from torchvision.models import resnet50

class MyFullyCustomModule(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = resnet50(weights='DEFAULT')
        self.model.fc = torch.nn.Linear(self.model.fc.in_features, 2)
        self.loss_fn = torch.nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images = batch['image']
        labels = batch['image_categories']
        loss = self.loss_fn(self(images), labels)
        self.log('train/loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        images = batch['image']
        labels = batch['image_categories']
        loss = self.loss_fn(self(images), labels)
        self.log('val/loss', loss)

    def test_step(self, batch, batch_idx):
        images = batch['image']
        labels = batch['image_categories']
        loss = self.loss_fn(self(images), labels)
        self.log('test/loss', loss)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)

trainer_full_custom = ImageClassificationTrainer(
    project=PROJECT_NAME,
    model=MyFullyCustomModule(),  # pass an instance
    image_size=480,
    batch_size=8,
    max_epochs=10,
    accelerator='auto',
)
results = trainer_full_custom.fit()
```

This still works with Datamint datasets, splits, MLflow logging, and checkpointing. The tradeoff is that a plain `LightningModule` does not automatically gain Datamint-native prediction methods for deployment. When you need Datamint-native serving, prefer `ClassificationModule`.

## 5. Deployment

The trainer **automatically** created and registered a deployment adapter in MLflow after training.
Deploy it directly to the Datamint server:

In [ ]:
job = api.deploy.start(
    model_name=PROJECT_NAME,
    model_alias="latest",
    with_gpu=False,
)

print(f"Deployment job started!")
print(f"Job ID: {job.id}")
print(f"Status: {job.status}")

In [ ]:
job = api.deploy.get_by_id(job.id)

print(f"Job Status: {job.status}")
print(f"Progress: {job.progress_percentage}%")

if job.error_message:
    print(f"Error: {job.error_message}")

### 5.1 Remote Inference

In [ ]:
r = api.resources.get_list(project_name=PROJECT_NAME, limit=1)[0]

inf_job = api.inference.submit(
    model_name=PROJECT_NAME,
    model_alias="latest",
    resource_id=r.id,
)
inf_job.wait()

print(inf_job.predictions)

In [ ]:
proj.show()

## Summary

| Step | Description |
|------|-------------|
| **Data Management** | Uploaded 4,000+ images to Datamint with tags and annotations |
| **Splits** | Created 80/10/10 train/val/test splits with `dataset.split().save()` |
| **Model Training** | Fine-tuned EfficientNetV2-S with `EfficientNetV2Trainer` — one line |
| **Experiment Tracking** | Metrics and model automatically logged with MLflow |
| **Custom Model** | Swapped in ResNet-18 by subclassing `ClassificationModule` |
| **Deployment** | Model automatically registered and deployed to Datamint server |

### See Also

- [Datamint Documentation](https://sonanceai.github.io/datamint-python-api/)
- [FracAtlas Paper](https://doi.org/10.1038/s41597-023-02432-4)
- [PyTorch Lightning Guide](https://lightning.ai/docs/pytorch/stable/)
- [MLflow Model Registry](https://mlflow.org/docs/latest/model-registry.html)